# Curating COVID example data


North American & European Measles D8 sequences >14KB 2017-01-01 to 2025-01-01 from [Pathoplexus](https://pathoplexus.org/measles/search?dataUseTerms=OPEN&lengthFrom=14000&orderBy=sampleCollectionDate&order=descending&genotype=D8&sampleCollectionDateRangeLowerFrom=2017-01-01&sampleCollectionDateRangeUpperTo=2025-01-01&geoLocCountry=Canada&geoLocCountry=USA&geoLocCountry=France&geoLocCountry=Italy&geoLocCountry=Austria&geoLocCountry=Hungary&geoLocCountry=Netherlands) on 206-02-17.

In [1]:
import pandas as pd
from Bio import SeqIO

In [2]:
metadata = pd.read_csv("measles_metadata_2026-02-17T1633.tsv", sep="\t", parse_dates=["sampleCollectionDate"])
metadata.head()

,accessionVersion,genotype,sampleCollectionDate,geoLocCountry,geoLocAdmin1,authors,authorAffiliations,hostNameScientific,length,earliestReleaseDate,...,biosampleAccession,completeness,geoLocAdmin2,geoLocCity,insdcAccessionFull,insdcRawReadsAccession,isLabHost,specimenCollectorSampleId,totalAmbiguousNucs,totalUnknownNucs
0,PP_003HBTY.2,D8,2017-11,Italy,NaN,"Marinelli, K.; Caucci, S.; Vincenzi, C.; Ferre...","Virologia, Ao Ospedali Riuniti Di Ancona",Homo sapiens,15872,2018-06-06,...,NaN,0.998616,NaN,NaN,MH173047.1,NaN,NaN,MVs/Ancona.ITA/45.17/1/[D8],0,0
1,PP_003JJWH.2,D8,2017-03,Italy,"Veneto region, Padova","Pacenti, M.; Maione, N.; Lavezzo, E.; Franchin...","University of Padua, Department of Molecular M...",Homo sapiens,15872,2019-12-12,...,NaN,0.998616,NaN,NaN,MK513623.1,NaN,NaN,MVs/Padova.ITA/13.17/1[D8],0,0
2,PP_003JJXF.2,D8,2017-04,Italy,"Veneto region, Padova","Pacenti, M.; Maione, N.; Lavezzo, E.; Franchin...","University of Padua, Department of Molecular M...",Homo sapiens,15872,2019-12-12,...,NaN,0.998616,NaN,NaN,MK513624.1,NaN,NaN,MVs/Padova.ITA/14.17/2[D8],0,0
3,PP_003JJYD.2,D8,2017-04,Italy,"Veneto region, Padova","Pacenti, M.; Maione, N.; Lavezzo, E.; Franchin...","University of Padua, Department of Molecular M...",Homo sapiens,15872,2019-12-12,...,NaN,0.998616,NaN,NaN,MK513625.1,NaN,NaN,MVs/Padova.ITA/14.17/3[D8],0,0
4,PP_003JJZB.2,D8,2017-05,Italy,"Veneto region, Verona","Pacenti, M.; Maione, N.; Lavezzo, E.; Franchin...","University of Padua, Department of Molecular M...",Homo sapiens,15872,2019-12-12,...,NaN,0.998616,NaN,NaN,MK513626.1,NaN,NaN,MVs/Verona.ITA/19.17/2[D8],0,0


In [3]:
metadata["sampleCollectionDate"].value_counts(dropna=False)

sampleCollectionDate
2019          29
2019-05       20
2019-07       16
2018-01        8
2024-02-11     8
              ..
2024-08-16     1
2024-10-04     1
2024-10-15     1
2024-03-29     1
2024-04-13     1
Name: count, Length: 199, dtype: int64

In [4]:
def safe_date_parser(x):
    return pd.to_datetime(x, errors='coerce', format="%Y-%m-%d")
metadata["sampleCollectionDate"] = metadata["sampleCollectionDate"].apply(safe_date_parser)
metadata["sampleCollectionDate"].value_counts(dropna=False)

sampleCollectionDate
NaT           141
2024-02-11      8
2024-02-29      6
2024-02-04      6
2024-02-13      5
             ... 
2024-08-16      1
2024-10-04      1
2024-10-15      1
2024-03-29      1
2024-04-13      1
Name: count, Length: 175, dtype: int64

In [5]:
metadata = metadata.dropna(subset=["sampleCollectionDate"])
metadata["sampleCollectionDate"].value_counts(dropna=False)
metadata.to_csv("all_measles_metadata.tsv", sep="\t", index=False)
valid_ids = metadata['accessionVersion'].tolist()


In [6]:
metadata['geoLocCountry'].value_counts()

geoLocCountry
Austria        127
Canada          62
Netherlands     54
Italy           14
Hungary          5
France           1
Name: count, dtype: int64

In [7]:
metadata['geoLocCountry'].unique()

<StringArray>
['Canada', 'Hungary', 'Netherlands', 'Italy', 'France', 'Austria']
Length: 6, dtype: str

In [8]:
north_american_metadata = metadata.query("geoLocCountry in ['USA', 'Canada', 'Mexico']")
north_american_metadata.to_csv("north_american_measles_metadata.tsv", sep="\t", index=False)
north_american_metadata_ids = north_american_metadata['accessionVersion'].tolist()


In [9]:
european_metadata = metadata.query("geoLocCountry in ['Italy', 'France', 'Hungary', 'Netherlands', 'Austria']")
european_metadata.to_csv("european_measles_metadata.tsv", sep="\t", index=False)
european_metadata_ids = european_metadata['accessionVersion'].tolist()

In [ ]:
for sequence in SeqIO.parse("measles_aligned-nuc_2026-02-17T1631.fasta", "fasta"):
    if sequence.id in valid_ids:
        with open("all_measles.fasta", "w") as all_file:
            SeqIO.write(sequence, all_file, "fasta")
    if sequence.id in north_american_metadata_ids:
        with open("north_american_measles.fasta", "w") as na_file:
            SeqIO.write(sequence, na_file, "fasta")
    if sequence.id in european_metadata_ids:
        with open("european_measles.fasta", "w") as eu_file:
            SeqIO.write(sequence, eu_file, "fasta")